# Stereo Visual Odometry with GTSAM

This notebook demonstrates a simple 3D stereo visual odometry problem using the GTSAM Python wrapper. The scenario is as follows:

1.  A robot starts at the origin (Pose 1: `X(1)`).
2.  The robot moves approximately 1 meter forward (Pose 2: `X(2)`).
3.  From both poses, the robot observes three landmarks (`L(1)`, `L(2)`, `L(3)`) with a stereo camera.

We will:
*   Define camera calibration and noise models.
*   Create a factor graph representing the problem.
*   Provide initial estimates for poses and landmark positions.
*   Optimize the graph using Levenberg-Marquardt to find the most probable configuration.


GTSAM Copyright 2010-2025, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/gtsam-project/gtsam-examples/blob/main/python/StereoVOExample_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import gtsam
import numpy as np

# For shorthand for common GTSAM types (like X for poses, L for landmarks)
from gtsam.symbol_shorthand import X, L


## 1. Setup Factor Graph and Priors

We start by creating an empty `NonlinearFactorGraph`.

The first pose `X(1)` is assumed to be at the origin. We'll add a hard constraint (prior) for this using `NonlinearEqualityPose3`. In GTSAM, factor keys are typically integers. We use `X(1)` as a symbolic key for the first pose.


In [20]:
# Create an empty nonlinear factor graph
graph = gtsam.NonlinearFactorGraph()

# Define the first pose at the origin
first_pose = gtsam.Pose3() # Default constructor is identity

# Add a prior on the first pose (X1). This constraint implies X1 is fixed at the origin.
# NonlinearEqualityPose3 is a factor that enforces X(1) == first_pose.
# It effectively acts as a prior with infinite precision (zero noise).
graph.add(gtsam.NonlinearEqualityPose3(X(1), first_pose))

print("Graph size after adding prior:", graph.size())


Graph size after adding prior: 1


## 2. Define Camera Calibration and Measurement Noise

- **Noise Model:** We define an isotropic noise model for the stereo measurements. `Isotropic.Sigma(3, 1.0)` means a 3-dimensional noise (for $u_L, u_R, v$) with a standard deviation of 1 pixel for each dimension.
- **Camera Calibration ([`Cal3_S2Stereo`](/gtsam/geometry/doc/Cal3_S2Stereo.ipynb)):** We define a stereo calibration model with the following properties:
  - $f_x=f_y=1000$
  - $s=0$
  - $(u,v)=(320,\,240)$
  - $b=0.2$

In [21]:
# Define the stereo measurement noise model (isotropic, 1 pixel standard deviation)
measurement_noise = gtsam.noiseModel.Isotropic.Sigma(3, 1.0)

# Create stereo camera calibration object
K = gtsam.Cal3_S2Stereo(1000, 1000, 0, 320, 240, 0.2)

## 3. Add Stereo Factors

We now add stereo factors to the graph. Each factor connects a camera pose to a landmark.
The `GenericStereoFactor3D` takes:
*   `measured`: The `StereoPoint2` measurement $(u_L, u_R, v)$.
*   `model`: The noise model for the measurement.
*   `poseKey`: Key for the camera pose.
*   `landmarkKey`: Key for the landmark.
*   `K`: The stereo camera calibration.

Landmark keys in the C++ example are 3, 4, 5. We'll use `L(1)`, `L(2)`, `L(3)` to map to these.

**Measurements from Pose 1 (`X(1)`)**


In [22]:
# Factors from X(1) to landmarks
# StereoPoint2(uL, uR, v)
graph.add(gtsam.GenericStereoFactor3D(
    gtsam.StereoPoint2(520, 480, 440), measurement_noise, X(1), L(1), K))
graph.add(gtsam.GenericStereoFactor3D(
    gtsam.StereoPoint2(120, 80, 440), measurement_noise, X(1), L(2), K))
graph.add(gtsam.GenericStereoFactor3D(
    gtsam.StereoPoint2(320, 280, 140), measurement_noise, X(1), L(3), K))

print("Graph size after adding X(1) factors:", graph.size())


Graph size after adding X(1) factors: 4


**Measurements from Pose 2 (`X(2)`)**


In [23]:
# Factors from X(2) to landmarks
graph.add(gtsam.GenericStereoFactor3D(
    gtsam.StereoPoint2(570, 520, 490), measurement_noise, X(2), L(1), K))
graph.add(gtsam.GenericStereoFactor3D(
    gtsam.StereoPoint2(70, 20, 490), measurement_noise, X(2), L(2), K))
graph.add(gtsam.GenericStereoFactor3D(
    gtsam.StereoPoint2(320, 270, 115), measurement_noise, X(2), L(3), K))

print("Graph size after adding all factors:", graph.size())

Graph size after adding all factors: 7


## 4. Create Initial Estimates

Nonlinear optimization requires initial estimates for all variables (poses and landmarks).
We create a `Values` object to store these.

*   `X(1)`: At origin (matches the prior).
*   `X(2)`: Robot moves forward, slightly off-axis: `Pose3(Rot3(), Point3(0.1, -0.1, 1.1))`.
*   Landmarks (`L(1)`, `L(2)`, `L(3)` or `3,4,5`): Placed at estimated 3D positions.


In [24]:
initial_estimate = gtsam.Values()

# Initial estimate for X(1)
initial_estimate.insert(X(1), first_pose)

# Initial estimate for X(2) - robot moved forward ~1m, with some small error
# Pose3(rotation, translation)
# gtsam.Rot3() is identity rotation
initial_pose2 = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(0.1, -0.1, 1.1))
initial_estimate.insert(X(2), initial_pose2)

# Initial estimates for landmark positions (Point3)
initial_estimate.insert(L(1), gtsam.Point3(1.0, 1.0, 5.0))
initial_estimate.insert(L(2), gtsam.Point3(-1.0, 1.0, 5.0))
initial_estimate.insert(L(3), gtsam.Point3(0.0, -0.5, 5.0))


## 5. Optimize the Factor Graph

We use the Levenberg-Marquardt optimizer to find the solution that minimizes the sum of squared errors defined by the factors in the graph, starting from the `initial_estimate`.


In [25]:
# Create a Levenberg-Marquardt optimizer
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_estimate)

# Optimize the graph
result = optimizer.optimize()

## 6. Analyze Results

We can extract the optimized poses and landmark positions from the `result` Values object.

For example, the true second pose (if the robot moved exactly 1m forward along Z from origin) would be `Pose3(Rot3(), Point3(0, 0, 1.0))`.
The true landmark positions can be triangulated from the perfect measurements and true poses.
The example's measurements and initial estimates are designed to be somewhat noisy/imperfect, so the optimizer refines them.

Let's check the error before and after optimization.


In [26]:
# print(result)
print(f"Initial Error: {graph.error(initial_estimate):.4f}")
print(f"Final Error  : {graph.error(result):.4f}")

# Extract and print optimized values for clarity
optimized_pose1 = result.atPose3(X(1))
optimized_pose2 = result.atPose3(X(2))
optimized_lm1 = result.atPoint3(L(1))
optimized_lm2 = result.atPoint3(L(2))
optimized_lm3 = result.atPoint3(L(3))

print("\nOptimized Pose X(1):\n", optimized_pose1)
print("\nOptimized Pose X(2):\n", optimized_pose2)
print(f"Translation component of X(2): {optimized_pose2.translation()}")

print("\nOptimized Landmark (L(1)):\n", optimized_lm1)
print("\nOptimized Landmark (L(2)):\n", optimized_lm2)
print("\nOptimized Landmark (L(3)):\n", optimized_lm3)

Initial Error: 3434.6236
Final Error  : 0.0000

Optimized Pose X(1):
 R: [
	1, 0, 0;
	0, 1, 0;
	0, 0, 1
]
t: 0 0 0


Optimized Pose X(2):
 R: [
	1, -3.03422e-17, -2.0943e-16;
	3.03422e-17, 1, 1.30264e-15;
	2.0943e-16, -1.30264e-15, 1
]
t:  6.11092e-13 -6.15462e-13            1

Translation component of X(2): [ 6.11092279e-13 -6.15461900e-13  1.00000000e+00]

Optimized Landmark (L(1)):
 [1. 1. 5.]

Optimized Landmark (L(2)):
 [-1.  1.  5.]

Optimized Landmark (L(3)):
 [-1.21564099e-16 -5.00000000e-01  5.00000000e+00]
